In [ ]:
import sys
import os
import io
import contextlib

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs, extract_history,
    plot_all_diagnostics, get_optimizer_colors,
    DIAG_KEYS_METRIC_SCALE, DIAG_KEYS_LEARNABLE,
    DIAG_KEYS_CURVATURE, DIAG_KEYS_TRAINING, DIAG_KEYS_OFFDIAG,
    TASK_CONFIGS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

BACKEND = "local"
RESULTS_DIR = os.path.join('..', '..', 'results')

DIAG_KEYS = (
    DIAG_KEYS_METRIC_SCALE + DIAG_KEYS_LEARNABLE +
    DIAG_KEYS_CURVATURE + DIAG_KEYS_TRAINING + DIAG_KEYS_OFFDIAG
)


def _quiet(fn, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)


def show_diagnostics(task_key):
    """Load and plot diagnostics for a sweep task."""
    cfg = TASK_CONFIGS[task_key]
    colors = get_optimizer_colors(cfg['optimizers'])
    best_runs = _quiet(load_best_runs,
        backend=BACKEND, optimizers=cfg['optimizers'],
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        metric_key=cfg['metric_key'], direction=cfg['direction'],
        sort_metric=cfg['sort_metric'], sort_order=cfg['sort_order'],
        iteration=cfg['iteration'],
    )
    diag_data = _quiet(extract_history, BACKEND, best_runs, DIAG_KEYS)
    figs = plot_all_diagnostics(diag_data, title_prefix=cfg['display_name'], colors=colors)
    for fig, name in figs:
        plt.show()
    if not figs:
        print(f"No diagnostic data for {cfg['display_name']}.")

# Optimizer Diagnostics

Internal state diagnostics (requires `--diagnostics` flag during sweep).

## MNIST MLP

In [ ]:
show_diagnostics("mnist_mlp")

## CIFAR-10 ResNet-18

In [ ]:
show_diagnostics("cifar10_resnet18")

## Shakespeare MiniGPT

In [ ]:
show_diagnostics("shakespeare_minigpt")

## Regression

In [ ]:
show_diagnostics("regression")

## Small Examples

In [ ]:
FUNCTIONS = ['beale', 'rosenbrock', 'himmelblau', 'ackley', 'rastrigin', 'styblinski_tang']
ITERATION_SE = 4

if BACKEND == "local":
    _avail = set()
    for fn in FUNCTIONS:
        td = os.path.join(RESULTS_DIR, f"small_examples_{fn}")
        if os.path.isdir(td):
            _avail.update(d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d)))
    SE_OPTIMIZERS = sorted(_avail)
else:
    SE_OPTIMIZERS = ['adam', 'sgd_learn_diag', 'sgd_learn_diag_curv']

se_colors = get_optimizer_colors(SE_OPTIMIZERS)

for fn in FUNCTIONS:
    best_runs = _quiet(load_best_runs,
        backend=BACKEND, optimizers=SE_OPTIMIZERS,
        task_tag=f"small_examples_{fn}", results_dir=RESULTS_DIR,
        metric_key="sweep_metric", direction="minimize",
        sort_metric="sweep_metric", sort_order="+", iteration=ITERATION_SE,
    )
    if not best_runs:
        continue
    diag_data = _quiet(extract_history, BACKEND, best_runs, DIAG_KEYS)
    figs = plot_all_diagnostics(diag_data, title_prefix=fn, colors=se_colors)
    for fig, name in figs:
        plt.show()
    if not figs:
        print(f"No diagnostic data for {fn}.")